# 🧠 Schizophrenia Recognition GNN+CEFAM — Tier 4 on Google Colab

Notebook này hướng dẫn chạy huấn luyện mô hình **Tier 4 (Spatiotemporal GNN + CEFAM)** sử dụng GPU miễn phí trên Google Colab từ kho lưu trữ GitHub của bạn.

## 🛠 Bước 1: Cấu hình Môi trường & Clone Repository
Chọn **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU** trước khi chạy.

In [ ]:
# 1. Clone repository từ GitHub của bạn
!git clone https://github.com/haimayoi/eye-movement-based-schizophrenia-recognition.git
%cd eye-movement-based-schizophrenia-recognition

In [ ]:
# 2. Cài đặt các thư viện cần thiết (bao gồm PyTorch Geometric)
!pip install torch-geometric
!pip install pyyaml omegaconf optuna catboost pyarrow openpyxl xgboost lightgbm

## 📂 Bước 2: Chuẩn bị dữ liệu EMS
Bạn cần đưa thư mục dữ liệu `EMS/` lên Colab. Có 2 cách chọn:

### Cách A: Upload file nén `EMS.zip` trực tiếp lên Colab
1. Nén thư mục `EMS` ở máy của bạn thành file `EMS.zip`.
2. Kéo thả file `EMS.zip` vào bảng điều khiển Files của Colab (nằm ở thư mục `/content/`).
3. Chạy cell giải nén dưới đây:

In [ ]:
# Chạy lệnh này nếu bạn chọn Cách A (upload file zip lên /content/)
# !unzip -q /content/EMS.zip -d /content/eye-movement-based-schizophrenia-recognition/

### Cách B: Mount Google Drive (Khuyên Dùng)
1. Tải thư mục `EMS` lên Google Drive của bạn (ví dụ đặt tại đường dẫn `MyDrive/EMS`).
2. Chạy cell dưới đây để Mount Drive và tạo liên kết (symlink) trực tiếp vào thư mục dự án:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo liên kết thư mục EMS từ Google Drive vào dự án Colab
!ln -s "/content/drive/MyDrive/EMS" ./EMS

## 🚀 Bước 3: Chạy Tiền xử lý & Trích xuất Đặc trưng (Tốc độ cực nhanh)
Vì môi trường Colab mới chưa có sẵn các tệp Parquet và đặc trưng trung gian, chúng ta sẽ chạy chuỗi lệnh này để tạo lại chúng (mất khoảng 2-3 phút).

In [ ]:
# 1. Sinh category mapping kích thích
!python -m src.utils.generate_category_map

# 2. Tiền xử lý lọc không-thời gian (Excel -> Parquet)
!python -m src.tier1_preprocessing.preprocess

# 3. Trích xuất đặc trưng trial (Stimulus-level)
!python -m src.tier2_features.stimulus_features

# 4. Trích xuất đặc trưng delta ngữ cảnh (Subject-level)
!python -m src.tier2_features.subject_aggregator

## 🔷 Bước 4: Xây dựng Đồ thị Scanpath (PyG Graphs)
Tác vụ này chuyển đổi các chuỗi mắt nhãn cầu thành đồ thị GNN và tự động sinh dummy RINet features 1056-dim để khớp nối luồng.

In [ ]:
!python -m src.tier4_advanced.graph_builder

## 🔥 Bước 5: Huấn luyện Mô hình GNN + CEFAM Hybrid Model
Sử dụng GPU T4 của Google Colab để huấn luyện mô hình hybrid với 4-Fold CV.

In [ ]:
!python scripts/train_tier4.py --config configs/cefam_config.yaml

## 📥 Bước 6: Lưu kết quả huấn luyện
Sao chép các checkpoints và file kết quả thu được về Google Drive để lưu trữ lâu dài.

In [ ]:
# Tạo thư mục lưu kết quả trên Google Drive
!mkdir -p "/content/drive/MyDrive/SZ_Recognition_Results/checkpoints"

# Sao chép các tệp checkpoint (.pt) và file kết quả summary (.json, .csv)
!cp -r results/checkpoints/* "/content/drive/MyDrive/SZ_Recognition_Results/checkpoints/"
!cp results/cefam_results_summary.json "/content/drive/MyDrive/SZ_Recognition_Results/"
!cp results/cefam_subject_val_predictions.csv "/content/drive/MyDrive/SZ_Recognition_Results/"